<a href="https://colab.research.google.com/github/rishh19/FlyRank-AI-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install duckdb huggingface_hub pandas scikit-learn

In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print(" Token loaded successfully!")

 Token loaded successfully!


In [3]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN
)

print(path)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [4]:
import duckdb

con = duckdb.connect()

con.sql(f"""
SELECT *
FROM read_parquet('{path}')
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [5]:
query = f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet('{path}')
"""

con.sql(query).df()

,total_rows
0,9841378


# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rishh19/FlyRank-AI-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Build the Feature Vector

The feature vector is built using search-performance metrics that are available before making a prediction.

Selected features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position

Missing values are handled by replacing them with zero for this educational analysis. No client identifiers, URLs, or private queries are included.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
query = f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet('{path}')
LIMIT 10000
"""

df = con.sql(query).df()

df = df.fillna(0)

print("Feature Vector")
print(df.head())

print("\nShape:", df.shape)

Feature Vector
   gsc_impressions  gsc_clicks  gsc_avg_position
0               20           0          3.350000
1                1           0          0.000000
2              125           1          4.928000
3                7           0          4.000000
4               11           0          2.272727

Shape: (10000, 3)


## Feature Notes

| Feature | Meaning | Missing Values | Available Before Prediction |
|---------|---------|----------------|-----------------------------|
| gsc_impressions | Number of search impressions | Filled with 0 | Yes |
| gsc_clicks | Number of search clicks | Filled with 0 | Yes |
| gsc_avg_position | Average search position | Filled with 0 | Yes |

These features are directly observed search-performance metrics and are suitable for exploratory modeling.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Missing Values")

print(df.isnull().sum())

print("\nData Types")

print(df.dtypes)

Missing Values
gsc_impressions     0
gsc_clicks          0
gsc_avg_position    0
dtype: int64

Data Types
gsc_impressions       int64
gsc_clicks            int64
gsc_avg_position    float64
dtype: object


## The Leakage Hunt

The selected features were reviewed for possible data leakage.

Checks performed:

- No label-derived columns were included.
- No future-window information was used.
- No product-specific flags were used.

Observed result: no intentional feature leakage was identified.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

print("Leakage Audit")
print("----------------")

for feature in features:
    print("✓", feature)

print("\nResult")
print("No future information detected.")
print("No target-derived columns detected.")
print("No product flags detected.")

Leakage Audit
----------------
✓ gsc_impressions
✓ gsc_clicks
✓ gsc_avg_position

Result
No future information detected.
No target-derived columns detected.
No product flags detected.


## What I Excluded and Why

The following information was intentionally excluded:

- Client names — protects privacy.
- URLs — avoids exposing sensitive information.
- Private search queries — excluded for confidentiality.
- Future-window information — prevents data leakage.
- Label-derived fields — avoids unrealistic model performance.

These choices help keep the analysis suitable for public sharing and decision-support.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded = {
    "Client Names": "Privacy",
    "URLs": "Confidentiality",
    "Private Queries": "Privacy",
    "Future Data": "Leakage Prevention",
    "Label-derived Columns": "Leakage Prevention"
}

print("Excluded Fields\n")

for field, reason in excluded.items():
    print(f"{field} -> {reason}")

Excluded Fields

Client Names -> Privacy
URLs -> Confidentiality
Private Queries -> Privacy
Future Data -> Leakage Prevention
Label-derived Columns -> Leakage Prevention


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.